Databricks pyspark project

Retail Sales Project

Business Requirements:
we have superstore sample retail data and our stakeholder wants to track the below Business metrics

- How many total number of customers we have ?

- total number of orders we have

- Total number of sales as of now

- Total profit

- Top sales by country

- Most profitable region and country

- Top sales category products

- Top 10 sales sub category products

- Most ordered quantity product

- Top customer based on sales , city

Note : dashboard/Notebook should be refresh based on weekly and monthly basis

In [0]:
dbutils.widgets.dropdown("time_period","Weekly",["Monthly","Weekly"])

In [0]:
from datetime import date, timedelta, datetime
from pyspark.sql.functions import *

time_period = dbutils.widgets.get("time_period")
print(time_period)
today = date.today()

if time_period == 'Weekly':
    start_date = today - timedelta(days=today.weekday(), weeks=1) - timedelta(days=1)
    end_date = start_date + timedelta(days=6)
else:
    first = today.replace(day=1)
    end_date = first - timedelta(days=1)
    start_date = first - timedelta(days=end_date.day)

print(start_date, end_date)


In [0]:
df = spark.read.csv("/Volumes/development/data/files/orders/superstore.csv",header=True,inferSchema=True)
display(df)

In [0]:
df.createOrReplaceTempView("sample")

In [0]:
%sql
select * from sample

In [0]:
%sql
select count(distinct Customer_id) from sample

In [0]:
display(spark.sql(f""" select count(distinct Customer_id) from sample
where order_date between '{start_date}' and  '{end_date}' """))

In [0]:
%sql
select count(distinct order_id) from sample

In [0]:
display(spark.sql(f""" select count(distinct order_id) from sample
where order_date between '{start_date}' and  '{end_date}' """))

In [0]:
%sql
select sum(try_cast(sales as double))as total_sales ,sum(try_cast(profit as double))as total_profit from sample

In [0]:
%sql
select sum(try_cast(sales as double))as total_sales , country from sample
group by 2

In [0]:
%sql
select sum(try_cast(sales as double))as total_sales ,region, country from sample
group by 2,3
order by 1 desc

In [0]:
%sql
select sum(try_cast(sales as double))as total_sales ,category from sample
group by 2
order by 1 desc

In [0]:
%sql
select sum(try_cast(sales as double))as total_sales, sub_category from sample
group by 2
order by 1 desc limit 10

In [0]:
%sql
select sum(try_cast(quantity as double))as total_quantity, product_name from sample
group by 2
order by 1 desc

In [0]:
%sql
select sum(try_cast(sales as double))as total_sales, customer_name,city from sample
group by 2 ,3
order by 1 desc